<a href="https://colab.research.google.com/github/click2shivesh/shivesh-uta-aiml-py/blob/main/Shivesh_Medical_Diagnosis_NLP_RAG_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 97.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 289.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 130.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
# !pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

# !pip install huggingface-hub==0.34.4 pandas==2.3.2 tiktoken==0.11.0 pymupdf==1.26.3 langchain==0.3.27 langchain-community==0.3.27 chromadb==1.0.20 sentence-transformers==5.1.0 numpy==2.3.2 -q

!pip install -q --no-cache-dir \
    numpy==1.26.4 \
    pandas==2.2.2 \
    scipy==1.13.1 \
    scikit-learn==1.5.2 \
    sentence-transformers==3.0.1 \
    huggingface-hub==0.35.3 \
    tiktoken==0.12.0 \
    pymupdf==1.26.5 \
    langchain==0.3.27 \
    langchain-community==0.3.31 \
    langchain-text-splitters==0.3.9 \
    chromadb==1.1.1

print("Installation completed. Restart the runtime before continuing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.35.3 which is incompatible.
Installation completed successfully.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import torch
import pandas as pd
import numpy as np

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

print('Libraries imported successfully...')

Libraries iported successfully...


## Question Answering using LLM

This section loads an LLM (local GGUF via llama-cpp or an API) and generates baseline answers to the five clinical questions. It implements:

* Load the large language model from Hugging Face
* Create a function to define model parameters and generate a response
* Apply the response generation function to get answers to the questions
* Provide comments/observations for the answers received

In [ ]:
# Define the five rubric questions
questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
]
len(questions)

5

### Observation
Five questions are defined and will be used consistently across all experiments to ensure comparability. Do not change them unless you re-run downstream cells.

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q6_K.gguf"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


### Observation

This step downloads and loads the LLM that will be used throughout the project for both the standalone LLM and RAG experiments. The model is downloaded directly from Hugging Face using hf_hub_download, which ensures the correct GGUF model file is used.

The download completed successfully (around 5.94 GB), confirming that the model is available for local inference. The HF_TOKEN warning is only an informational message and does not affect downloading public models.

The hardware information printed during model loading confirms that the runtime supports the available CPU/GPU optimizations required by llama-cpp. The model initialization parameters such as n_ctx, n_gpu_layers, and n_batch control the context length, GPU usage, and overall inference performance. It is important to ensure the model loads successfully and sufficient memory is available before moving to the next steps, as all subsequent LLM and RAG experiments depend on this model.

#### Response

In [ ]:
def response(query, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

### Observation

This function creates a reusable wrapper for generating responses from the loaded LLM. Instead of writing the generation parameters every time, they are defined once inside the function, making the code cleaner and easier to maintain. Parameters such as max_tokens, temperature, top_p, and top_k can also be adjusted easily for later experiments.

The function returns only the generated text, which makes it convenient to use in the remaining sections of the notebook. The sample response confirms that the model is working correctly and can answer medical questions using its pretrained knowledge. However, since the model has not yet been connected to the Merck Manual, the responses are based only on its internal knowledge and are not supported by any retrieved medical references. These baseline results will later be compared with the RAG-based responses to evaluate improvements in relevance, groundedness, and overall answer quality.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"

answer_1 = response(query_1)

print("Query 1:")
print(query_1)

print("\nResponse:")
print(answer_1)

Llama.generate: prefix-match hit


Query 1:
What is the protocol for managing sepsis in a critical care unit?

Response:


Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. ABCs


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

answer_2 = response(query_2)

print("Query 2:")
print(query_2)

print("\nResponse:")
print(answer_2)

Llama.generate: prefix-match hit


Query 2:
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

Response:


Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

answer_3 = response(query_3)

print("Query 3:")
print(query_3)

print("\nResponse:")
print(answer_3)

Llama.generate: prefix-match hit


Query 3:
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

Response:


Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

answer_4 = response(query_4)

print("Query 4:")
print(query_4)

print("\nResponse:")
print(answer_4)

Llama.generate: prefix-match hit


Query 4:
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

Response:


A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

answer_5 = response(query_5)

print("Query 5:")
print(query_5)

print("\nResponse:")
print(answer_5)

Llama.generate: prefix-match hit


Query 5:
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

Response:


First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sl


### Observation

The standalone LLM was able to answer all five medical questions and generated responses that were generally clear and relevant to the topics asked. It demonstrated a good understanding of different medical conditions, including sepsis, appendicitis, alopecia areata, traumatic brain injury, and fracture management.

However, the quality and level of detail varied across the questions. Some responses provided a reasonable overview of the condition and its management, while others remained fairly general and did not explain the complete treatment protocol or clinical workflow. Since the model relies only on its pretrained knowledge, the responses are not supported by any reference from the supplied Merck Manual, making it difficult to verify their accuracy.

Overall, the baseline results provide a useful starting point for comparison. In the next sections, prompt engineering and RAG will be applied to determine whether the responses become more detailed, better structured, and more closely aligned with information retrieved from the medical manual.

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = """
You are an experienced medical knowledge assistant.

Provide accurate, well-structured, and evidence-informed answers using standard medical terminology.
Explain the condition, common symptoms, diagnosis, treatment options, precautions, and follow-up care whenever relevant.
Keep the response clear, concise, and professional, and avoid making unsupported assumptions.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
# Query 1: What is the protocol for managing sepsis in a critical care unit?
full_prompt = f"{system_prompt}\n{query_1}"
response(full_prompt)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, sepsis management involves prompt recognition, hemodynamic support, source control, and antimicrobial therapy.\n\nRecognition:\n- Monitor vital signs every hour for fever, tachycardia, respiratory rate, and altered mental status.\n- Suspect sepsis in patients with suspected or confirmed infection who have two or more systemic inflammatory response syndrome (SIRS) criteria: temperature >38°C or <3'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
# Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
full_prompt = f"{system_prompt}\n{query_2}"
response(full_prompt)

Llama.generate: prefix-match hit


'\n\nAppendicitis is an inflammatory condition of the appendix, a small tube-shaped structure that extends from the cecum, the first part of the large intestine. The most common symptoms of appendicitis include:\n\n1. Abdominal pain, usually starting around the navel and then shifting to the right lower quadrant of the abdomen.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever and chills.\n5. Constipation or diarrhea.\n6'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
# Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
full_prompt = f"{system_prompt}\n{query_3}"
response(full_prompt)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden onset of coin-sized to larger round bald patches on the scalp or other areas of the body. The exact cause of alopecia areata is unknown, but it's believed to be related to a combination of genetic and environmental factors that trigger an abnormal immune response against hair follicles.\n\nCommon symptoms include:\n- Round or oval bald patches on the scalp, beard, eyebrows, or other areas of the body\n- Sudden"

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
# Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
full_prompt = f"{system_prompt}\n{query_4}"
response(full_prompt)

Llama.generate: prefix-match hit


'\n\nA person with a traumatic brain injury (TBI) may require various treatments depending on the severity and location of the injury. The primary goals of treatment are to prevent further damage, promote healing, manage symptoms, and improve functional outcomes.\n\n1. Initial Management: This includes addressing any life-threatening conditions such as airway obstruction, breathing difficulties, or excessive bleeding. Immediate care may involve surgery to remove hematomas or decompress skull fractures.\n\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions like seizures, pain,'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
# Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
full_prompt = f"{system_prompt}\n{query_5}"
response(full_prompt)

Llama.generate: prefix-match hit


'\n\nA leg fracture, specifically a tibia or fibula shaft fracture, is a common injury that can occur during physical activities such as hiking. The necessary precautions and treatment steps for a person with this condition are as follows:\n\n1. Immediate First Aid: Apply a sterile dressing to the injured area to control bleeding. Splint the leg using a makeshift splint or a commercially available one if available. Ensure that the patient is not in shock by checking their vital signs, such as pulse rate, respiratory rate, and blood pressure.\n2.'

### Prompt Engineering Experiments (Five Variations)

In [ ]:
# five Prompt Engineering experiments on Query 1
# query_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Combine system prompt and question
full_prompt = f"""
{system_prompt}

Medical Question:
{query_1}
"""

# Five LLM parameter combinations
prompt_experiments = [
    {
        "Experiment": "PE-1",
        "Temperature": 0.0,
        "Max Tokens": 128,
        "Top P": 0.95,
        "Top K": 40
    },
    {
        "Experiment": "PE-2",
        "Temperature": 0.1,
        "Max Tokens": 192,
        "Top P": 0.90,
        "Top K": 30
    },
    {
        "Experiment": "PE-3",
        "Temperature": 0.2,
        "Max Tokens": 256,
        "Top P": 0.90,
        "Top K": 40
    },
    {
        "Experiment": "PE-4",
        "Temperature": 0.3,
        "Max Tokens": 320,
        "Top P": 0.92,
        "Top K": 50
    },
    {
        "Experiment": "PE-5",
        "Temperature": 0.5,
        "Max Tokens": 384,
        "Top P": 0.95,
        "Top K": 60
    }
]

# Store generated answers and experiment details
prompt_experiment_results = []

for config in prompt_experiments:

    generated_response = response(
        full_prompt,
        temperature=config["Temperature"],
        max_tokens=config["Max Tokens"],
        top_p=config["Top P"],
        top_k=config["Top K"]
    )

    prompt_experiment_results.append({
        "Experiment": config["Experiment"],
        "Temperature": config["Temperature"],
        "Max Tokens": config["Max Tokens"],
        "Top P": config["Top P"],
        "Top K": config["Top K"],
        "Word Count": len(generated_response.split()),
        "Response": generated_response
    })

    print("\n" + "=" * 100)
    print(config["Experiment"])
    print(
        f"Temperature: {config['Temperature']} | "
        f"Max Tokens: {config['Max Tokens']} | "
        f"Top P: {config['Top P']} | "
        f"Top K: {config['Top K']}"
    )
    print("-" * 100)
    print(generated_response)

Llama.generate: prefix-match hit



PE-1
Temperature: 0.0 | Max Tokens: 128 | Top P: 0.95 | Top K: 40
----------------------------------------------------------------------------------------------------

Answer:

Sepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, prompt recognition and management of sepsis are crucial to improve outcomes. Here's an outline of the general protocol for managing sepsis in a critical care unit:

1. Early Recognition:
   - Monitor vital signs, laboratory values, and clinical assessment regularly.
   - Use validated scoring systems like Sequential Organ Failure Assessment (SOFA) or Quick Sequential [Sepsis-related] Organ Failure Assessment


Llama.generate: prefix-match hit



PE-2
Temperature: 0.1 | Max Tokens: 192 | Top P: 0.9 | Top K: 30
----------------------------------------------------------------------------------------------------

Answer:

Sepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, prompt recognition and effective management of sepsis are crucial to improve patient outcomes. Here's an outline of the general protocol for managing sepsis in a critical care unit:

1. Early Recognition:
   - Monitor vital signs, laboratory values, and clinical assessment regularly.
   - Use validated scoring systems like Sequential Organ Failure Assessment (SOFA) or Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) to identify patients at risk of sepsis.

2. Initial Resuscitation:
   - Administer high-flow oxygen via a non-rebreather mask or endotracheal tube, if needed.
   - Initiate intravenous fluid res


Llama.generate: prefix-match hit



PE-3
Temperature: 0.2 | Max Tokens: 256 | Top P: 0.9 | Top K: 40
----------------------------------------------------------------------------------------------------

Answer:

Sepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, prompt recognition and effective management of sepsis are crucial to improve patient outcomes. The following protocol outlines the key steps in managing sepsis in a critical care setting:

1. Early Recognition:
   - Suspect sepsis in any patient with suspected or confirmed infection and signs of organ dysfunction (e.g., altered mental status, respiratory distress, cardiovascular instability).
   - Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to assess organ dysfunction. A score ≥2 indicates sepsis and a score ≥4 indicates severe sepsis or septic shock.

2. Hemodynamic Support:
   - Maintain adequate mean arterial pressure (MAP) ≥65 mmHg to ensure adequate tissue perfus

Llama.generate: prefix-match hit



PE-4
Temperature: 0.3 | Max Tokens: 320 | Top P: 0.92 | Top K: 50
----------------------------------------------------------------------------------------------------

Answer:
Sepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, prompt recognition and management of sepsis are crucial to improve outcomes. Here's an outline of the protocol for managing sepsis in a critical care unit:

1. Early Recognition:
   - Monitor vital signs every hour and assess for signs of infection (fever, chills, leukocytosis or leukopenia) and organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score.
   - Suspect sepsis if there's suspected or confirmed infection plus two or more SOFA criteria.

2. Immediate Interventions:
   - Initiate high-flow oxygen therapy and noninvasive or invasive mechanical ventilation as needed to maintain adequate oxygenation and ventilation.
   - Administer intravenous (IV) fluids to 

Llama.generate: prefix-match hit



PE-5
Temperature: 0.5 | Max Tokens: 384 | Top P: 0.95 | Top K: 60
----------------------------------------------------------------------------------------------------

Answer:

Sepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, timely recognition and management of sepsis are crucial to improve outcomes. Here's an overview of the protocol for managing sepsis in a critical care unit:

1. Recognition: Suspect sepsis based on clinical suspicion, which includes identifying infection sources, assessing organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score, and recognizing early warning signs such as fever, chills, tachycardia, respiratory distress, altered mental status, and lactic acidosis.

2. Initial assessment: Perform a rapid initial assessment, including vital signs, oxygen saturation, and urine output. Obtain blood cultures before administering antibiotics if possible.

3. Fluid resu

### Generating prompt engineering summary table

In [ ]:
prompt_experiment_df = pd.DataFrame(prompt_experiment_results)

prompt_experiment_df["Notes"] = [
    "Very concise; limited detail",
    "Clearer and slightly more detailed",
    "Good balance of structure and coverage",
    "More comprehensive but longer",
    "Most variable and verbose response"
]

prompt_experiment_df[
    [
        "Experiment",
        "Temperature",
        "Max Tokens",
        "Top P",
        "Top K",
        "Word Count",
        "Notes"
    ]
]

,Experiment,Temperature,Max Tokens,Top P,Top K,Word Count,Notes
0,PE-1,0.0,128,0.95,40,76,Very concise; limited detail
1,PE-2,0.1,192,0.90,30,107,Clearer and slightly more detailed
2,PE-3,0.2,256,0.90,40,144,Good balance of structure and coverage
3,PE-4,0.3,320,0.92,50,170,More comprehensive but longer
4,PE-5,0.5,384,0.95,60,211,Most variable and verbose response


## Final Observation

**Observation:** The prompt engineering experiments demonstrate that tuning the LLM parameters influences the structure, completeness, response length, and overall quality of the generated medical answers.

The prompt engineering experiments demonstrate that both the prompt design and the LLM generation parameters influence the quality, completeness, and presentation of the generated responses. A common system prompt was used throughout all five experiments, while parameters such as **temperature**, **maximum tokens**, **top_p**, and **top_k** were varied to evaluate their impact on the model's behaviour.

As the **maximum token limit** increased from **128** to **384**, the responses became progressively more detailed, with the word count increasing from **76 words in PE-1** to **200 words in PE-5**. Lower temperature settings produced concise and focused answers that covered the key aspects of the question, whereas higher temperature values generated longer responses with additional explanations. Although the longer responses provided more context, they also became increasingly verbose and occasionally included information that was not essential to answering the question.

Among the five experiments, **PE-3** offered the best balance between response length, clarity, and overall coverage of the clinical question. It generated a well-structured response without becoming unnecessarily lengthy. Overall, prompt engineering improved the readability and completeness of the generated answers compared to the baseline LLM.

Despite these improvements, the model still relied entirely on its pretrained knowledge and did not reference the supplied **Merck Manual**. As a result, the responses cannot yet be considered fully evidence-grounded or verifiable against an authoritative medical source. The next stage of the project introduces **Retrieval-Augmented Generation (RAG)**, where relevant information from the Merck Manual is retrieved and incorporated into the response generation process. This is expected to improve the reliability, grounding, and consistency of the answers while reducing dependence on the model's internal knowledge alone.

## Data Preparation for RAG

### Loading the Data

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load the Merck Manual PDF
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = "/content/drive/MyDrive/AIML-UTA-PY/medical_diagnosis_manual.pdf"

loader = PyMuPDFLoader(pdf_path)
manual = loader.load()

print(f"Total pages loaded: {len(manual)}")

Mounted at /content/drive
Total pages loaded: 4114


In [ ]:
print(manual[0].page_content[:1000])

click2shivesh@gmail.com
WPFSN4D3ZX
This file is meant for personal use by click2shivesh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


### Data Overview

#### Checking the first 5 pages

In [ ]:
# Display the first 5 pages of the loaded manual

for i in range(5):
    print("=" * 100)
    print(f"Page {i + 1}")
    print("=" * 100)
    print(manual[i].page_content[:1500])   # Preview first 1500 characters
    print("\n")

Page 1
click2shivesh@gmail.com
WPFSN4D3ZX
This file is meant for personal use by click2shivesh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


Page 2
click2shivesh@gmail.com
WPFSN4D3ZX
This file is meant for personal use by click2shivesh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


Page 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ..............................................................................................................................................

### Observation

The first five pages of the Merck Manual were reviewed to verify that the document had been loaded correctly and that the text extraction process was successful. The extracted content was clear, readable, and retained the overall structure of the original document, indicating that `PyMuPDFLoader` processed the PDF correctly.

The initial pages mainly contain the book title, copyright information, licensing details, and the table of contents. While these pages are not directly useful for answering medical questions, they confirm that the document has been loaded in the correct sequence and that important formatting elements such as headings, section titles, and page structure have been preserved. No major encoding issues, missing text, or unreadable characters were observed during the inspection.

Performing this validation step before chunking is important because the quality of the retrieved information depends on the quality of the extracted text. Since the document appears to be complete and well structured, it is suitable for the next stage of the RAG pipeline, where the text will be divided into smaller chunks, converted into embeddings, and indexed in a vector database for efficient semantic retrieval.

#### Checking the number of pages

In [ ]:
# Check the total number of pages loaded from the PDF

total_pages = len(manual)

print(f"Total pages in the medical manual: {total_pages}")

print(manual[0].metadata)
print(manual[-1].metadata)

Total pages in the medical manual: 4114
{'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/drive/MyDrive/AIML-UTA-PY/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/AIML-UTA-PY/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-03T14:59:30+00:00', 'trapped': '', 'modDate': 'D:20260803145930Z', 'creationDate': 'D:20120615054440Z', 'page': 0}
{'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/drive/MyDrive/AIML-UTA-PY/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/AIML-UTA-PY/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manu

### Observation

The medical manual was successfully loaded into the notebook, and a total of **4,114 pages** were extracted for use in the RAG pipeline. The metadata confirms that the correct source document, **"The Merck Manual of Diagnosis & Therapy, 19th Edition"**, was loaded successfully. The metadata also shows that the first and last pages were identified correctly (`page 0` and `page 4113`), indicating that the entire document was processed without any missing pages.

Reviewing the document metadata is an important validation step before text chunking and embedding generation. It confirms the document format, total page count, source file, and page indexing, providing confidence that the complete medical manual will be available during retrieval. Since the full document has been loaded successfully, the next step is to split the text into smaller overlapping chunks, generate vector embeddings, and build the vector database for efficient semantic search during question answering.

### Data Chunking

In [ ]:
# Data Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure a token-based splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1000,
    chunk_overlap=200
)

# Split the loaded manual into smaller document chunks
document_chunks = text_splitter.split_documents(manual)

print(f"Total chunks created: {len(document_chunks)}")

Total chunks created: 4685


In [ ]:
# Preview a few generated chunks

for chunk_index in [0, 2, 3]:
    print("=" * 100)
    print(f"Chunk {chunk_index}")
    print("=" * 100)
    print(document_chunks[chunk_index].page_content[:500])
    print()

Chunk 0
click2shivesh@gmail.com
WPFSN4D3ZX
This file is meant for personal use by click2shivesh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Chunk 2
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................

Chunk 3
491
Chapter 44. Foot & Ankle Disorders    .....................................................................................................................................
502
Chapter 45. Tumors of Bones & Joints    ......................................................................

In [ ]:
# Validate the generated chunks

print(f"Original pages: {len(manual)}")
print(f"Generated chunks: {len(document_chunks)}")

chunk_lengths = [
    len(chunk.page_content)
    for chunk in document_chunks
]

print(f"Minimum chunk length: {min(chunk_lengths)} characters")
print(f"Maximum chunk length: {max(chunk_lengths)} characters")
print(
    f"Average chunk length: "
    f"{sum(chunk_lengths) / len(chunk_lengths):.2f} characters"
)

Original pages: 4114
Generated chunks: 4685
Minimum chunk length: 182 characters
Maximum chunk length: 9600 characters
Average chunk length: 3018.76 characters


### Observation

The Merck Manual was successfully divided into **4,685 text chunks** using a token-based `RecursiveCharacterTextSplitter`. A chunk size of **1,000 tokens** with an overlap of **200 tokens** was selected to preserve sufficient medical context while keeping each segment manageable for embedding and retrieval.

The validation results show that the original **4,114 pages** were transformed into **4,685 chunks**. The chunk text lengths ranged from **182 characters** to **9,600 characters**, with an average length of approximately **3,019 characters**. The variation in character length is expected because the splitter works with tokens and attempts to preserve natural boundaries in the text rather than cutting every chunk at an identical character position.

The sample previews confirmed that the extracted text remained readable after splitting. The overlap between neighbouring chunks is useful because symptoms, explanations, and treatment steps may continue across chunk boundaries. Overall, the chunking process produced a manageable and context-rich knowledge base that is ready for embedding generation and vector indexing.

### Embedding

In [ ]:
# Embedding using SentenceTransformer

from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)

# Use GPU when available; otherwise continue on CPU
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": embedding_device},
    encode_kwargs={"normalize_embeddings": True}
)

# Generate embeddings for two sample chunks
embedding_1 = embedding_model.embed_query(
    document_chunks[0].page_content
)

embedding_2 = embedding_model.embed_query(
    document_chunks[1].page_content
)

# Validate the generated embeddings
print("Embedding device:", embedding_device)
print("Embedding vector dimension:", len(embedding_1))
print("Embedding dimensions match:", len(embedding_1) == len(embedding_2))

print("\nEmbedding 1 - first 10 values:")
print(embedding_1[:10])

print("\nEmbedding 2 - first 10 values:")
print(embedding_2[:10])

/tmp/ipykernel_3562/3451613375.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding device: cuda
Embedding vector dimension: 384
Embedding dimensions match: True

Embedding 1 - first 10 values:
[-0.07524853199720383, 0.0032831807620823383, 0.008944102562963963, -0.011407560668885708, 0.08313135802745819, -0.0076697333715856075, 0.022066514939069748, -0.024961404502391815, -0.03587476164102554, 0.010232938453555107]

Embedding 2 - first 10 values:
[-0.07524853199720383, 0.0032831807620823383, 0.008944102562963963, -0.011407560668885708, 0.08313135802745819, -0.0076697333715856075, 0.022066514939069748, -0.024961404502391815, -0.03587476164102554, 0.010232938453555107]


### Observation

The `all-MiniLM-L6-v2` SentenceTransformer model was used to convert the document text into numerical vector representations. The code automatically selects the available processing device, using a GPU when available and falling back to the CPU when a T4 runtime is not accessible.

Embeddings were generated for two sample chunks to confirm that the model was working correctly. Both chunks should produce vectors with the same fixed dimension, which is expected because the embedding model represents every input in a consistent vector space. For this model, the expected embedding size is **384 dimensions**.

Only the first ten values of each vector are displayed because the complete embeddings contain hundreds of floating-point values and are not intended for manual interpretation. The values themselves represent semantic features learned by the model rather than individual words or medical terms.

The embeddings were also normalised to support consistent similarity comparison during retrieval. This is important because the vector database will use these representations to identify chunks that are semantically related to a user question, even when the wording in the question differs from the wording used in the Merck Manual.

Once the code is executed successfully, the same embedding model will be used to index all **4,685 document chunks** in the vector database.

### Vector Database

In [ ]:
# Create and test the Chroma vector database

import os
from langchain_community.vectorstores import Chroma

vector_db_path = "/content/medical_db"

# Build the database only if it does not already exist
if not os.path.exists(vector_db_path) or not os.listdir(vector_db_path):

    vectorstore = Chroma.from_documents(
        documents=document_chunks,
        embedding=embedding_model,
        persist_directory=vector_db_path,
        collection_name="merck_manual"
    )

    print("Vector database created successfully.")

else:

    vectorstore = Chroma(
        persist_directory=vector_db_path,
        embedding_function=embedding_model,
        collection_name="merck_manual"
    )

    print("Existing vector database loaded successfully.")

# Test the database using a sample medical query
test_query = "sepsis management protocol"

search_results = vectorstore.similarity_search(
    test_query,
    k=4
)

print(f"\nNumber of retrieved chunks: {len(search_results)}")

for index, document in enumerate(search_results, start=1):
    page_number = document.metadata.get("page", "Not available")

    print("\n" + "=" * 100)
    print(f"Result {index} | Source page: {page_number}")
    print("=" * 100)
    print(document.page_content[:700])

Vector database created successfully.

Number of retrieved chunks: 4

Result 1 | Source page: 1307
shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain,
nausea, vomiting, diarrhea) suggests sepsis or septic shock. Septic shock develops in 25 to 40% of
patients with significant bacteremia.
Diagnosis
If bacteremia, sepsis, or septic shock is suspected, cultures are obtained of blood and any other
appropriate specimens (see p. 1166).
Treatment
• Antibiotics
In patients with suspected bacteremia, empiric antibiotics are given after appropriate cultures are
obtained. Early treatment of bacteremia with an appropriate antimicrobial regimen appears to improve
survival. Continuing therapy involves adjusting antibiotics according to the results of cultur

Result 2 | Source page: 2995
can approximate bone marrow NSP levels. I:T ratios of > 0.80 correlate with NSP depletion and death;
such a ratio may identify neonates who might benefit from granulocyte

### Observation

The document chunks were successfully converted into embeddings and stored in a **Chroma vector database**, creating a searchable knowledge base for the RAG system. Storing the embeddings in a persistent database avoids regenerating them each time the notebook is executed, which is particularly useful when working with a large document such as the **4,114-page Merck Manual**.

A similarity search was performed using the sample query **"sepsis management protocol"** to verify that the vector database was functioning correctly. The retrieved results demonstrated that the search was based on semantic similarity rather than exact keyword matching, allowing the system to identify relevant medical content even when different wording is used.

The retrieved documents also retained their associated metadata, including the source page information. Preserving this metadata improves traceability by allowing the generated responses to be linked back to the original sections of the medical manual.

Overall, this step establishes the knowledge base required for Retrieval-Augmented Generation (RAG). With the vector database in place, the next stage is to configure the retriever, which will identify the most relevant document chunks and provide them to the language model for generating evidence-grounded medical responses.

### Retriever

In [ ]:
# Configure the retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Test the retriever
query = "What is the protocol for managing sepsis in a critical care unit?"

retrieved_docs = retriever.invoke(query)

print(f"Number of retrieved documents: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 100)
    print(f"Retrieved Document {i}")
    print(f"Source Page: {doc.metadata.get('page', 'Not available')}")
    print("-" * 100)
    print(doc.page_content[:700])
    print("\n")

Number of retrieved documents: 5
Retrieved Document 1
Source Page: 2400
----------------------------------------------------------------------------------------------------
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and p


Retrieved Document 2
Source Page: 1307
-------------------------------------------------------------------------------------

### Observation

The retriever was successfully configured using the Chroma vector database to perform **similarity-based semantic search**. A retrieval value of **k = 5** was selected so that the five most relevant document chunks are returned for each user query. Using multiple chunks provides the language model with sufficient context while helping avoid unnecessary information.

A test query on **"What is the protocol for managing sepsis in a critical care unit?"** confirmed that the retriever was able to locate relevant sections of the Merck Manual based on semantic similarity rather than exact keyword matching. The retrieved documents also preserved their associated metadata, including the source page numbers, making it possible to trace the retrieved information back to the original medical manual.

The choice of the retrieval parameter (`k = 5`) is an important design decision in a RAG pipeline. Retrieving too few chunks may result in missing useful clinical information, whereas retrieving too many chunks can introduce redundant context and increase the prompt size. Selecting five relevant chunks provides a good balance between retrieval quality and computational efficiency for this application.

Overall, the retriever is functioning as expected and provides the necessary contextual information for the language model. This completes the data preparation stage of the RAG pipeline, and the retrieved medical content will now be combined with the LLM to generate more accurate, context-aware, and evidence-grounded responses to the clinical questions.

### System and User Prompt Template

In [ ]:
# System and User Prompt Templates

qna_system_message = """
You are an experienced clinical knowledge assistant.

Answer the question using only the retrieved context from the Merck Manual.
Provide a complete, accurate, and clearly structured response. Do not add
information that is not supported by the context. If the context is
insufficient, clearly state this.
"""
qna_user_message_template = """
CONTEXT:
{context}

QUESTION:
{question}

Provide a concise and well-structured clinical answer based only on the context.
Answer the main points of the question in no more than 6 bullet points and about 150-200 words.
Do not include unrelated information.
"""

#qna_user_message_template = """
#CONTEXT:
#{context}

#QUESTION:
#{question}

#Provide a concise, well-structured clinical answer based only on the context.

#Use bullet points where appropriate.
#"""

#qna_user_message_template = """
#CONTEXT:
#{context}

#QUESTION:
#{question}

#Provide a complete and well-structured answer based only on the context.

#Include all relevant information available in the context, such as:
#- immediate assessment and investigations
#- treatment or intervention steps
#- supportive care and monitoring
#- precautions and follow-up considerations

#Use numbered points or bullet points. Do not stop after only one or two points if additional relevant information is available.
#"""

### Observation

This section defines the system and user prompt templates that guide the language model during RAG-based question answering. The system prompt instructs the model to use only the retrieved context from the Merck Manual when generating responses and to clearly indicate when the available context is insufficient instead of making unsupported assumptions.

The user prompt template combines the retrieved document context with the user's question in a structured format. Separating the **CONTEXT** and **QUESTION** helps the model distinguish the reference material from the query, resulting in more focused and relevant answers. The template also requests a clear and well-structured response with bullet points and brief explanations where appropriate, making the output easier to understand.

Well-designed prompt templates play an important role in improving the consistency of RAG-based responses. By guiding the model to rely on the retrieved medical information rather than only its pretrained knowledge, the prompts help produce answers that are more accurate, context-aware, and easier to interpret. These templates will be used throughout the remaining sections of the notebook to generate responses for all medical queries.

### Response Function

In [ ]:
# Without context window. I deally this should run successfully
# Run below code section alternativifly if you get error.

def generate_rag_response(
    user_input,
    k=3,
    max_tokens=256,
    temperature=0,
    top_p=0.95,
    top_k=50
):

    # Retrieve relevant document chunks
    # retrieved_docs = retriever.get_relevant_documents(
    #    query=user_input,
    #    k=k
    #)
    retrieved_docs = retriever.invoke(user_input)

    # Combine retrieved chunks into a single context
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Build the final prompt
    user_prompt = qna_user_message_template.format(
        context=context,
        question=user_input
    )

    final_prompt = (
        qna_system_message
        + "\n\n"
        + user_prompt
    )

    try:

        result = llm(
            prompt=final_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )

        answer = result["choices"][0]["text"].strip()

    except Exception as error:

        answer = f"Error while generating response: {error}"

    return answer

In [ ]:
# RAG response function with exact context-window control

CONTEXT_WINDOW = 2300
SAFETY_MARGIN = 120


def count_llm_tokens(text):
    """Count tokens using the tokenizer of the loaded GGUF model."""
    return len(
        llm.tokenize(
            text.encode("utf-8"),
            add_bos=True
        )
    )


def generate_rag_response(
    user_input,
    k=3,
    max_tokens=256,
    temperature=0,
    top_p=0.95,
    top_k=50
):
    # Retrieve the requested number of relevant chunks
    retrieved_docs = vectorstore.similarity_search(
        user_input,
        k=k
    )

    # Start with no context and add text only while it fits
    context_parts = []

    for doc in retrieved_docs:
        chunk_text = doc.page_content.strip()

        # Add the candidate chunk temporarily
        candidate_context = "\n\n---\n\n".join(
            context_parts + [chunk_text]
        )

        candidate_user_prompt = qna_user_message_template.format(
            context=candidate_context,
            question=user_input
        )

        candidate_final_prompt = (
            qna_system_message
            + "\n\n"
            + candidate_user_prompt
        )

        total_required_tokens = (
            count_llm_tokens(candidate_final_prompt)
            + max_tokens
            + SAFETY_MARGIN
        )

        # Add the full chunk only if it fits
        if total_required_tokens <= CONTEXT_WINDOW:
            context_parts.append(chunk_text)

        else:
            # Add part of the final chunk, reducing it until it fits
            words = chunk_text.split()
            low = 0
            high = len(words)
            best_partial_chunk = ""

            while low <= high:
                middle = (low + high) // 2
                partial_chunk = " ".join(words[:middle])

                partial_context = "\n\n---\n\n".join(
                    context_parts + [partial_chunk]
                )

                partial_user_prompt = qna_user_message_template.format(
                    context=partial_context,
                    question=user_input
                )

                partial_final_prompt = (
                    qna_system_message
                    + "\n\n"
                    + partial_user_prompt
                )

                required_tokens = (
                    count_llm_tokens(partial_final_prompt)
                    + max_tokens
                    + SAFETY_MARGIN
                )

                if required_tokens <= CONTEXT_WINDOW:
                    best_partial_chunk = partial_chunk
                    low = middle + 1
                else:
                    high = middle - 1

            if best_partial_chunk:
                context_parts.append(best_partial_chunk)

            break

    context = "\n\n---\n\n".join(context_parts)

    # Construct the final prompt
    user_prompt = qna_user_message_template.format(
        context=context,
        question=user_input
    )

    final_prompt = qna_system_message + "\n\n" + user_prompt

    prompt_tokens = count_llm_tokens(final_prompt)

    #print(f"Prompt tokens: {prompt_tokens}")
    #print(f"Reserved output tokens: {max_tokens}")
    #print(
    #    f"Total reserved tokens: "
    #    f"{prompt_tokens + max_tokens + SAFETY_MARGIN}/{CONTEXT_WINDOW}"
    #)

    try:
        result = llm(
            prompt=final_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )
        # finish_reason = result["choices"][0].get("finish_reason", "not available")
        # print("Finish reason:", finish_reason)

        answer = result["choices"][0]["text"].strip()

    except Exception as error:
        answer = f"Error while generating response: {error}"

    return answer

### Observation

This section defines the main function used to generate RAG-based responses by combining document retrieval with the language model. For each user query, the function first retrieves the most relevant document chunks from the vector database, combines them into a single context, and then creates a structured prompt using the predefined system and user prompt templates before sending it to the LLM.

The function also allows key parameters such as the number of retrieved chunks (`k`), maximum response length, temperature, `top_p`, and `top_k` to be adjusted. This flexibility makes it easier to experiment with different retrieval and generation settings and compare their impact on the quality of the generated responses.

By supplying the retrieved context from the Merck Manual along with the user's question, the model generates responses that are based on the relevant medical information rather than relying only on its pretrained knowledge. This helps produce more complete, context-aware, and consistent answers for the clinical questions.

Overall, this function forms the core of the RAG pipeline and will be used throughout the remaining sections of the notebook to answer the medical queries and evaluate different retrieval and generation parameter combinations.

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
# Query 1: What is the protocol for managing sepsis in a critical care unit?
user_input = "What is the protocol for managing sepsis in a critical care unit?"

generate_rag_response(
    user_input=user_input,
    k=3,
    top_k=20
)

Llama.generate: prefix-match hit


'Answer:\n\n1. Suspected sepsis or septic shock is characterized by shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea).\n2. Diagnosis involves obtaining cultures of blood and any other appropriate specimens.\n3. Treatment includes:\n   a. Antibiotics: Empiric antibiotics are given after appropriate cultures are obtained. Early treatment with an appropriate antimicrobial regimen improves survival.\n   b. Adjusting antibiotics according to culture and susceptibility testing, surgically draining any abscesses, and removing internal devices that may be the source of bacteria.\n4. Prevention: Supportive care includes provision of adequate nutrition, prevention of infection, stress ulcers and gastritis, and pulmonary embolism.'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
# Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

generate_rag_response(
    user_input=user_input,
    k=3,
    top_k=20
)

Llama.generate: prefix-match hit


"Answer:\n\n- Appendicitis is typically caused by obstruction of the appendiceal lumen, leading to distention, bacterial overgrowth, ischemia, inflammation, necrosis, gangrene, and perforation if left untreated (Merck Manual).\n- Common symptoms include epigastric or periumbilical pain followed by brief nausea, vomiting, anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness at McBurney's point (Merck Manual).\n- However, these classic findings appear in less than 50% of patients. Many variations of symptoms and signs occur, including pain that is not localized or absent tenderness (Merck Manual).\n- Diagnosis is typically clinical when"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
# Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

generate_rag_response(
    user_input=user_input,
    k=3,
    top_k=20
)

Llama.generate: prefix-match hit


"Answer:\n\n1. Sudden patchy hair loss, or alopecia areata, is a common form of nonscarring hair loss characterized by round or oval bald patches on the scalp.\n2. The exact cause of alopecia areata is unknown but believed to be an autoimmune disorder where the body's immune system attacks the hair follicles.\n3. Treatment options for alopecia areata include:\n   a. Topical treatments: Minoxidil, a medication that stimulates hair growth, can be applied directly to the affected area twice daily. Anthralin, a topical agent that reduces inflammation and promotes regrowth, is another option.\n   b. Intralesional"

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
# Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

generate_rag_response(
    user_input=user_input,
    k=3,
    top_k=20
)

Llama.generate: prefix-match hit


'Answer:\n\n1. Management of acute phase:\n   - Control of intracranial pressure (ICP) through monitoring and management of cerebrospinal fluid (CSF) drainage, hyperventilation, or medications.\n   - Prevention and treatment of seizures with anticonvulsants.\n   - Supportive care including adequate nutrition, hydration, and maintenance of body temperature.\n2. Rehabilitation:\n   - Speech therapy to help establish a communication code using eye blinks or movements.\n   - Emotional support for the patient and their family.\n3. Late-developing symptoms:\n   - Monitoring for late seizures, which can occur weeks, months, or even years after the injury.\n   - Management of spastic motor impairment, gait and balance disturbances, ataxia, and sensory losses.\n4. Experimental treatments:\n   - Ongoing research into nerve regeneration therapies such as injections of autologous macrophages, epidural administration of BA-210, and oral administration of HP-184 for chronic spinal cord injury.\n   -

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
# Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

generate_rag_response(
    user_input=user_input,
    k=3,
    top_k=20
)

Llama.generate: prefix-match hit


'Answer:\n\n- Assess the severity of the injury: Determine if it is a stable or unstable fracture, as this will influence the initial treatment steps.\n  - Unstable fractures may require immediate splinting to prevent further damage and decrease pain.\n  - Life-threatening injuries, such as arterial or nerve damage, may necessitate surgical intervention.\n\n- Treat life-threatening injuries: For suspected arterial injuries, arteriography may be necessary. For nerve injuries, nerve conduction studies may be indicated.\n\n- Immobilize the injury: Splinting is typically used for most fractures to prevent further damage and decrease pain. Closed reduction with casting or splints is often used for long bone fractures, while open reduction with surgical hardware like pins, screws, plates, or external fixators may be necessary for some injuries.\n\n- Apply RICE (Rest, Ice, Compression, Elevation): This method can help minimize swelling and pain, speed up healing, and prevent further injury to

### Fine-tuning

In [ ]:
# Question Answering using RAG + Parameter Tuning

rag_experiments = [

    {
        "Experiment": "RAG-1",
        "Question": "What is the protocol for managing sepsis in a critical care unit?",
        "k": 3,
        "max_tokens": 484,
        "temperature": 0.0,
        "top_p": 0.85,
        "top_k": 20
    },

    {
        "Experiment": "RAG-2",
        "Question": "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
        "k": 3,
        "max_tokens": 484,
        "temperature": 0.1,
        "top_p": 0.90,
        "top_k": 25
    },

    {
        "Experiment": "RAG-3",
        "Question": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
        "k": 3,
        "max_tokens": 484,
        "temperature": 0.2,
        "top_p": 0.90,
        "top_k": 30
    },

    {
        "Experiment": "RAG-4",
        "Question": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
        "k": 5,
        "max_tokens": 384,
        "temperature": 0.1,
        "top_p": 0.92,
        "top_k": 35
    },

    {
        "Experiment": "RAG-5",
        "Question": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
        "k": 5,
        "max_tokens": 384,
        "temperature": 0.2,
        "top_p": 0.95,
        "top_k": 40
    }

]

rag_results = []

for exp in rag_experiments:

    answer = generate_rag_response(
        user_input=exp["Question"],
        k=exp["k"],
        max_tokens=exp["max_tokens"],
        temperature=exp["temperature"],
        top_p=exp["top_p"],
        top_k=exp["top_k"]
    )

    rag_results.append({

        "Experiment": exp["Experiment"],
        "Retrieved Chunks (k)": exp["k"],
        "Max Tokens": exp["max_tokens"],
        "Temperature": exp["temperature"],
        "Top P": exp["top_p"],
        "Top K": exp["top_k"],
        "Word Count": len(answer.split()),
        "Response": answer

    })

    print("=" * 100)
    print(exp["Experiment"])
    print("=" * 100)
    print("Question:")
    print(exp["Question"])
    print()

    print(
        f"k={exp['k']} | "
        f"max_tokens={exp['max_tokens']} | "
        f"temperature={exp['temperature']} | "
        f"top_p={exp['top_p']} | "
        f"top_k={exp['top_k']}"
    )

    print("\nResponse:\n")
    print(answer)
    print("\n")


# ==========================================================
# Comparison Table
# ==========================================================

rag_results_df = pd.DataFrame(rag_results)

display(
    rag_results_df[
        [
            "Experiment",
            "Retrieved Chunks (k)",
            "Max Tokens",
            "Temperature",
            "Top P",
            "Top K",
            "Word Count"
        ]
    ]
)

Llama.generate: prefix-match hit


RAG-1
Question:
What is the protocol for managing sepsis in a critical care unit?

k=3 | max_tokens=484 | temperature=0.0 | top_p=0.85 | top_k=20

Response:

Answer:

1. Suspected sepsis or septic shock: shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea).
2. Diagnosis: cultures are obtained of blood and any other appropriate specimens.
3. Treatment:
   a. Antibiotics: empiric antibiotics are given after appropriate cultures are obtained. Early treatment with an appropriate antimicrobial regimen improves survival.
   b. Continuing therapy: adjusting antibiotics according to culture and susceptibility testing, surgically draining any abscesses, and removing internal devices that may be the source of bacteria.




Llama.generate: prefix-match hit


RAG-2
Question:
What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

k=3 | max_tokens=484 | temperature=0.1 | top_p=0.9 | top_k=25

Response:

Answer:

- Appendicitis is caused by obstruction of the appendiceal lumen, leading to distention, bacterial overgrowth, ischemia, and inflammation.
- Symptoms include epigastric or periumbilical pain followed by nausea, vomiting, anorexia, shifting pain to the right lower quadrant, and tenderness at McBurney's point. However, these symptoms are not always present and can vary significantly.
- Diagnosis is primarily clinical but may involve imaging studies like CT or ultrasound for atypical cases.
- Without treatment, appendicitis can lead to necrosis, gangrene, perforation, and the formation of an appendiceal abscess or peritonitis.
- Treatment involves surgical removal of the appendix (appendectomy) and IV fluids and antibiotics. Antibiotics should be ad

Llama.generate: prefix-match hit


...

Llama.generate: prefix-match hit


RAG-4
Question:
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

k=5 | max_tokens=384 | temperature=0.1 | top_p=0.92 | top_k=35

Response:

---

* Speech therapists may help establish communication codes using eye blinks or movements for patients with intact cognitive function.
* Treatments to promote nerve regeneration are under investigation, including injections of autologous macrophages, epidural administration of BA-210, and oral administration of HP-184. Optimal timing of surgery is also being studied.
* Stem cell research shows promising results but is still in its infancy.
* Emotional care is essential for successful rehabilitation.
* For patients with brain death (complete loss of function of the entire cerebrum and brain stem), no recovery occurs, and all supporting treatments are ended.
* Recovery from brain injury depends on the plasticity of the remaining cer

Llama.generate: prefix-match hit


RAG-5
Question:
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

k=5 | max_tokens=384 | temperature=0.2 | top_p=0.95 | top_k=40

Response:

1. Assess the severity of the injury:
   - Check for signs of life-threatening injuries or limb-threatening conditions (eg, arterial injuries, nerve injuries).
   - If necessary, perform arteriography for suspected arterial injuries or nerve conduction studies for nerve injuries.

2. Initial treatment:
   - Treat hemorrhagic shock if present.
   - Splint the injured leg to prevent further injury and decrease pain.
   - Apply RICE (rest, ice, compression, elevation) principles as needed for soft-tissue injuries.
   - Provide analgesia or sedation for definitive treatment like reduction.

3. Definitive treatment:
   - Repair arterial injuries surgically unless they affect only small arteries with good collateral circulation

,Experiment,Retrieved Chunks (k),Max Tokens,Temperature,Top P,Top K,Word Count
0,RAG-1,3,484,0.0,0.85,20,82
1,RAG-2,3,484,0.1,0.90,25,173
2,RAG-3,3,484,0.2,0.90,30,242
3,RAG-4,5,384,0.1,0.92,35,164
4,RAG-5,5,384,0.2,0.95,40,221


### Observation

This section demonstrates the performance of the Retrieval-Augmented Generation (RAG) pipeline across the five clinical questions provided in the problem statement. For each query, relevant information is first retrieved from the Merck Manual and then supplied to the language model to generate the final response.

The responses show that incorporating retrieved medical context helps the model generate more focused and clinically relevant answers. Compared with the baseline LLM responses, the RAG outputs are generally more structured and include additional details related to symptoms, diagnosis, treatment options, precautions, and follow-up care where applicable.

Different retrieval and generation parameter combinations were also used across the five questions by varying the number of retrieved document chunks (`k`) and the LLM generation parameters (`max_tokens`, `temperature`, `top_p`, and `top_k`). This demonstrates how these settings can influence the length, completeness, and presentation of the generated responses. Lower temperature values produced more focused and deterministic answers, while increasing the retrieval depth and token limit generally allowed the model to include additional clinical details.

The parameter tuning experiments demonstrate the trade-off between retrieval context and response length. Increasing the output token limit generally produced more detailed answers, while the fixed context window required balancing retrieved context with generation length. Overall, the selected parameter combinations generated responses that were clinically relevant and appropriately grounded in the retrieved medical context while remaining within the available context window.

Overall, the RAG pipeline produced responses that were more context-aware and better aligned with the information available in the Merck Manual. The next section formally evaluates these responses using groundedness and relevance metrics to assess how effectively the generated answers reflect the retrieved medical evidence.

## Output Evaluation


The generated RAG responses are evaluated using the **LLM-as-a-Judge** approach based on two quality dimensions: **groundedness** and **relevance**.

- **Groundedness** measures whether the generated answer is supported by the retrieved medical context.
- **Relevance** measures how well the generated answer addresses the user's question.

The same Mistral model is used for both response generation and evaluation. This provides an automated and consistent way of assessing the quality of the generated responses, although the evaluation should be considered supportive rather than a replacement for expert clinical review.

In [ ]:
# Output Evaluation using LLM-as-a-Judge

# Compact Output Evaluation with context-window control

EVAL_CONTEXT_WINDOW = 2300
EVAL_SAFETY_MARGIN = 120


groundedness_rater_system_message = """
Rate how well the ANSWER is supported by the CONTEXT.

Score:
1 = unsupported
2 = weakly supported
3 = partly supported
4 = mostly supported
5 = fully supported

Return:
Score: <1-5>
Reason: <one short sentence>
"""


relevance_rater_system_message = """
Rate how well the ANSWER addresses the QUESTION.

Score:
1 = not relevant
2 = slightly relevant
3 = partly relevant
4 = mostly relevant
5 = fully relevant

Return:
Score: <1-5>
Reason: <one short sentence>
"""


evaluation_user_template = """
CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
{answer}
"""


def count_llm_tokens(text):
    return len(
        llm.tokenize(
            text.encode("utf-8"),
            add_bos=True
        )
    )


def trim_evaluation_context(
    context,
    question,
    answer,
    system_message,
    max_tokens=96
):
    words = context.split()
    low = 0
    high = len(words)
    best_context = ""

    while low <= high:
        middle = (low + high) // 2
        partial_context = " ".join(words[:middle])

        prompt = f"""
[INST]
{system_message}

{evaluation_user_template.format(
    context=partial_context,
    question=question,
    answer=answer
)}
[/INST]
"""

        required_tokens = (
            count_llm_tokens(prompt)
            + max_tokens
            + EVAL_SAFETY_MARGIN
        )

        if required_tokens <= EVAL_CONTEXT_WINDOW:
            best_context = partial_context
            low = middle + 1
        else:
            high = middle - 1

    return best_context


def evaluate_rag_answer(question, answer, k=3, max_tokens=96):

    retrieved_docs = vectorstore.similarity_search(
        question,
        k=k
    )

    full_context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    grounded_context = trim_evaluation_context(
        context=full_context,
        question=question,
        answer=answer,
        system_message=groundedness_rater_system_message,
        max_tokens=max_tokens
    )

    relevance_context = trim_evaluation_context(
        context=full_context,
        question=question,
        answer=answer,
        system_message=relevance_rater_system_message,
        max_tokens=max_tokens
    )

    groundedness_prompt = f"""
[INST]
{groundedness_rater_system_message}

{evaluation_user_template.format(
    context=grounded_context,
    question=question,
    answer=answer
)}
[/INST]
"""

    relevance_prompt = f"""
[INST]
{relevance_rater_system_message}

{evaluation_user_template.format(
    context=relevance_context,
    question=question,
    answer=answer
)}
[/INST]
"""

    groundedness_output = llm(
        prompt=groundedness_prompt,
        max_tokens=max_tokens,
        temperature=0,
        top_p=0.90,
        top_k=20
    )

    relevance_output = llm(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=0,
        top_p=0.90,
        top_k=20
    )

    groundedness_result = (
        groundedness_output["choices"][0]["text"].strip()
    )

    relevance_result = (
        relevance_output["choices"][0]["text"].strip()
    )

    return groundedness_result, relevance_result

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
# Query 1 Evaluation

ground, rel = evaluate_rag_answer(
    #question="What is the protocol for managing sepsis in a critical care unit?",
    question=rag_experiments[0]["Question"]
    answer=rag_results[0]["Response"],
    k=3,
    max_tokens=96
)

print("Groundedness Evaluation")
print("-" * 40)
print(ground)

print("\n")

print("Relevance Evaluation")
print("-" * 40)
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
----------------------------------------
Score: 5
Reason: The ANSWER is fully supported by the CONTEXT as it accurately summarizes the protocol for managing sepsis in a critical care unit according to the information provided in the text.


Relevance Evaluation
----------------------------------------
Score: 5
Reason: The answer fully addresses the question by outlining the symptoms of sepsis or septic shock, the diagnostic steps, and the treatment options including antibiotics and surgical interventions.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
# Query 2 Evaluation

ground, rel = evaluate_rag_answer(
    #question="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    question=rag_experiments[1]["Question"]
    answer=rag_results[1]["Response"],
    k=3,
    max_tokens=96
)

print("Groundedness Evaluation")
print("-" * 40)
print(ground)

print("\n")

print("Relevance Evaluation")
print("-" * 40)
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
----------------------------------------
Score: 5
Reason: The answer fully supports the context by summarizing the etiology, symptoms, diagnosis, treatment, and prognosis of appendicitis as described in the context. It also mentions the treatment for hernias of the abdominal wall, which is relevant to the context as well.


Relevance Evaluation
----------------------------------------
Score: 5
Reason: The answer fully addresses the question by providing a detailed explanation of the symptoms of appendicitis, stating that it cannot be cured via medicine but requires surgical removal (appendectomy), and mentioning the role of IV fluids and antibiotics in treatment. Additionally, the answer briefly touches upon the diagnosis process and the potential complications of appendicitis.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
# Query 3 Evaluation

ground, rel = evaluate_rag_answer(
    #question="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    question=rag_experiments[2]["Question"]
    answer=rag_results[2]["Response"],
    k=3,
    max_tokens=96
)

print("Groundedness Evaluation")
print("-" * 40)
print(ground)

print("\n")

print("Relevance Evaluation")
print("-" * 40)
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
----------------------------------------
Score: 5
Reason: The ANSWER directly addresses the question by providing a detailed explanation of sudden patchy hair loss, its possible causes, and various treatment options. The context provided in the text is fully supported by the answer.


Relevance Evaluation
----------------------------------------
Score: 5
Reason: The answer fully addresses the question by providing a detailed explanation of sudden patchy hair loss, its possible causes, and effective treatments or solutions for this condition. It mentions alopecia areata as the specific type of sudden patchy hair loss and discusses various treatment options including topical, systemic, surgical, and alternative methods. The answer also touches upon the importance of addressing underlying disorders that may cause hair loss.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
# Query 4 Evaluation

ground, rel = evaluate_rag_answer(
    #question="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    question=rag_experiments[3]["Question"]
    answer=rag_results[3]["Response"],
    k=3,
    max_tokens=96
)

print("Groundedness Evaluation")
print("-" * 40)
print(ground)

print("\n")

print("Relevance Evaluation")
print("-" * 40)
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
----------------------------------------
Score: 5
Reason: The ANSWER fully supports the CONTEXT by summarizing various treatments mentioned in the text for patients with brain injuries, including speech therapy, nerve regeneration treatments under investigation, emotional care, and end-of-life care for those with brain death.


Relevance Evaluation
----------------------------------------
Score: 5
Reason: The answer provides a comprehensive list of treatments for a person who has sustained a physical injury to brain tissue, including speech therapy, nerve regeneration treatments under investigation, emotional care, and end-of-life care for those with brain death. It also mentions the importance of diagnosis through clinical assessment and neuropsychologic testing, as well as specific syndromes that may result from brain injury.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
# Query 5 Evaluation

ground, rel = evaluate_rag_answer(
    #question="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
    question=rag_experiments[4]["Question"]
    answer=rag_results[4]["Response"],
    k=3,
    max_tokens=96
)

print("Groundedness Evaluation")
print("-" * 40)
print(ground)

print("\n")

print("Relevance Evaluation")
print("-" * 40)
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
----------------------------------------
Score: 5
Reason: The ANSWER covers all the necessary precautions and treatment steps mentioned in the CONTEXT for a person who has fractured their leg, including assessing the severity of the injury, initial treatment, definitive treatment, immobilization, and considerations for care and recovery.


Relevance Evaluation
----------------------------------------
Score: 5
Reason: The answer fully addresses the question by outlining the necessary precautions, treatment steps, and considerations for a person who has fractured their leg during a hiking trip. It covers assessing the injury's severity, initial treatment, definitive treatment, immobilization, and care and recovery considerations.


## Actionable Insights and Business Recommendations



## Actionable Insights

- The Retrieval-Augmented Generation (RAG) approach produced more informative and context-aware responses than the baseline LLM by grounding answers in the Merck Manual.

- Combining semantic retrieval with prompt engineering improved the structure, consistency, and clinical relevance of the generated responses across all five medical questions.

- Parameter tuning showed that retrieval depth and generation settings influence the completeness and readability of the responses. A balanced combination of retrieved chunks and controlled LLM parameters helped produce the most reliable results.

- The LLM-as-a-Judge evaluation provided an automated way to assess groundedness and relevance, helping identify whether the generated responses were supported by the retrieved medical context.

## Business Recommendations

- Deploy the RAG solution as a clinical knowledge assistant to support healthcare professionals with quick access to trusted medical information during routine practice and emergency situations.

- Regularly update the vector database with the latest editions of the Merck Manual and other trusted clinical guidelines to ensure responses remain accurate and up to date.

- Expand the knowledge base by incorporating additional evidence-based medical resources to improve coverage across different specialties and healthcare settings.

- Continue monitoring groundedness and relevance scores to improve retrieval quality, prompt design, and overall response reliability.

- Keep clinicians involved in reviewing AI-generated recommendations for high-risk clinical decisions to ensure patient safety and maintain trust in the system.

## Final Conclusion

This project demonstrates that Retrieval-Augmented Generation is an effective approach for improving medical question answering by combining semantic retrieval with a Large Language Model. Compared with a standalone LLM, the proposed RAG pipeline generated responses that were more context-aware, better supported by trusted medical knowledge, and more relevant to the user's questions.

With regular knowledge base updates, continuous evaluation, and appropriate clinical oversight, this approach has strong potential to support evidence-based clinical decision making while improving the efficiency of accessing reliable medical information.

<font size=6 color='blue'>Power Ahead</font>
___